In [1]:
import torch
import re
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
def extract_sentences(input_text):

    sentences = re.split(r'[.!?]', input_text)
    return sentences

def tokenize_sentence(sentence):

    tokenized_sentence =  sentence.split()
    return tokenized_sentence

def text_to_indices(tokenized_sentences):

    indexed_sentences = []
    for sentence in tokenized_sentences:

        indexed_sentence = []
        for word in sentence:
            
            if word in vocab:
                indexed_sentence.append(vocab[word])
            else:
                indexed_sentence.append(vocab['<UNK>'])
        
        indexed_sentences.append(indexed_sentence)

    return indexed_sentences

input_text = "Visvesvaraya National Institute of Technology Nagpur (VNIT), formerly known as Visvesvaraya Regional College of Engineering (VRCE) is a public technical university located in the city of Nagpur, Maharashtra.[4] Established in 1960, the institute is among 31 National Institutes of Technology (NITs) in the country. In 2007, the institute was conferred with the status of Institute of National Importance by the National Institutes of Technology, Science Education and Research Act, 2007 of the Parliament of India with all other NITs. Formerly known as Visvesvaraya Regional College of Engineering (VRCE), the institute is named in honour of an eminent engineer, planner and statesman Sir M. Visvesvaraya. The Institute awards Bachelor's, Master's and Doctorate degrees in engineering, technology, architecture, science and humanities. The institute's history can be traced back to 1947 when the Architecture Department was established by the Madhya Pradesh Government. Following the second five-year plan (1956–60) in India, several industrial projects were contemplated. The Regional Engineering Colleges (RECs) were established by the central government to mimic the IITs at a regional level and act as benchmarks for the other colleges in that state. For the Western region in the year 1960, the institute was established under the name Visvesvaraya Regional College of Engineering (VRCE). It was established under the scheme sponsored by Govt. of India and Govt. of Maharashtra. The college was started in June 1960 by amalgamating the State Govt. Engineering College functioning in Nagpur since July 1956. In the meeting held in October 1962, the governing board of the college resolved to name it after the eminent engineer, planner, and statesman of the country M. Visvesvaraya. The college started functioning in 1960 from a camp office in the premises of Government Polytechnic, Nagpur and subsequently, an area of about 214 acres was acquired to house an independent Regional Engineering College at its present location. It has its jurisdiction over entire state of Maharashtra."
input_text = input_text.lower()

sentences = extract_sentences(input_text)
print(sentences)

tokenized_sentences = []

for sentence in sentences:

    if sentence == '':
        sentences.remove(sentence)

    else: 
        tokenized_sentence = tokenize_sentence(sentence)
        tokenized_sentences.append(tokenized_sentence)

print(tokenized_sentences)

vocab = {}
vocab['<UNK>'] = 0

# Create Vocabulary

for tokenized_sentence in tokenized_sentences:

    for word in tokenized_sentence:

        if word not in vocab:
            vocab[word] = len(vocab)


print(vocab)
print(len(vocab))

indexed_sentences = text_to_indices(tokenized_sentences)
print(indexed_sentences)

['visvesvaraya national institute of technology nagpur (vnit), formerly known as visvesvaraya regional college of engineering (vrce) is a public technical university located in the city of nagpur, maharashtra', '[4] established in 1960, the institute is among 31 national institutes of technology (nits) in the country', ' in 2007, the institute was conferred with the status of institute of national importance by the national institutes of technology, science education and research act, 2007 of the parliament of india with all other nits', ' formerly known as visvesvaraya regional college of engineering (vrce), the institute is named in honour of an eminent engineer, planner and statesman sir m', ' visvesvaraya', " the institute awards bachelor's, master's and doctorate degrees in engineering, technology, architecture, science and humanities", " the institute's history can be traced back to 1947 when the architecture department was established by the madhya pradesh government", ' followi

In [3]:
class CustomSentenceDataset(Dataset):

    def __init__(self, indexed_sentences):

        sampled_sentences = []

        for sentence in indexed_sentences:

            for i in range(1, len(sentence)):

                sampled_sentences.append((sentence[:i:], sentence[i]))

        self.sentences = sampled_sentences
    
    def __len__(self):

        return len(self.sentences) # returns number of training examples basically
    
    def __getitem__(self, index):

        return torch.tensor(self.sentences[index][0], dtype=torch.long), torch.tensor(self.sentences[index][1], dtype=torch.long)

training_dataset = CustomSentenceDataset(indexed_sentences)
training_dataloader = DataLoader(training_dataset, batch_size = 1, shuffle = True)

In [4]:
class LSTM_Architecture(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding_layer = nn.Embedding(vocab_size, 150)
        self.LSTM_layer = nn.LSTM(150, 100, batch_first = True)
        self.fc = nn.Linear(100, vocab_size)

    def forward(self, sampled_sentence):

        embedded_sentence = self.embedding_layer(sampled_sentence)
        all_hidden, (final_hidden, final_cell) = self.LSTM_layer(embedded_sentence)
        output = self.fc(final_hidden.squeeze(0))
        return output
    


In [5]:
# Just for figuring out dimensions, not a part of the next word predictor
input = training_dataset[4][0].unsqueeze(0)
target = training_dataset[4][1]
print(input.shape)
print(target)

x = nn.Embedding(11, 150)
y = nn.LSTM(150, 100, batch_first = True)
z = nn.Linear(100, 11)

a = x(input)
print(a.shape)

b, (c, d) = y(a)
print(b.shape)
print(c.shape)
print(d.shape)

e = z(d)
print(e.shape)

torch.Size([1, 5])
tensor(6)
torch.Size([1, 5, 150])
torch.Size([1, 5, 100])
torch.Size([1, 1, 100])
torch.Size([1, 1, 100])
torch.Size([1, 1, 11])


In [6]:
# Creating Model and Initializing Hyperparameters:

model = LSTM_Architecture(len(vocab))
model.to(device)

loss_function = nn.CrossEntropyLoss()

learning_rate = 0.0025
epochs = 25

optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [7]:
for epoch in range(epochs):

    total_loss = 0
    for sentence, target in training_dataloader:

        sentence, target = sentence.to(device), target.to(device)

        # Setting Optimizer Gradients to Zero
        optimizer.zero_grad()

        # Forward Pass:
        y_pred = model(sentence)

        # Loss Calculation:
        loss = loss_function(y_pred, target)
        total_loss = total_loss + loss.item()

        # Backpropagation:
        loss.backward()

        # Updation:
        optimizer.step()

    avg_epoch_loss = total_loss / len(training_dataloader)
    print(f"Epoch: {epoch + 1} | Loss: {avg_epoch_loss}")

Epoch: 1 | Loss: 4.921907586423126
Epoch: 2 | Loss: 3.291449961214754
Epoch: 3 | Loss: 1.7747559166494504
Epoch: 4 | Loss: 0.8417777670531634
Epoch: 5 | Loss: 0.4577567559065977
Epoch: 6 | Loss: 0.28638229873723575
Epoch: 7 | Loss: 0.20343840698164106
Epoch: 8 | Loss: 0.18175094650831777
Epoch: 9 | Loss: 0.1331952502085777
Epoch: 10 | Loss: 0.12451690930803935
Epoch: 11 | Loss: 0.11174296264692424
Epoch: 12 | Loss: 0.10218625951902878
Epoch: 13 | Loss: 0.0917411766257263
Epoch: 14 | Loss: 0.0936268204056211
Epoch: 15 | Loss: 0.08304910899342094
Epoch: 16 | Loss: 0.083146519732435
Epoch: 17 | Loss: 0.07897598886090752
Epoch: 18 | Loss: 0.07931165723690552
Epoch: 19 | Loss: 0.08338597747461864
Epoch: 20 | Loss: 0.08052963857364769
Epoch: 21 | Loss: 0.07896605967323773
Epoch: 22 | Loss: 0.07411764630843529
Epoch: 23 | Loss: 0.07186190581735384
Epoch: 24 | Loss: 0.07825423121018527
Epoch: 25 | Loss: 0.11769962024371469


In [8]:
test_text = "The College"
def prediction(test_text):

    # Tokenize Text
    test_text = test_text.lower()
    tokenized_sentence = tokenize_sentence(test_text)

    # Text to indices
    indexed_text = []

    for word in tokenized_sentence:

        if word in vocab:
            indexed_text.append(vocab[word])
        else:
            indexed_text.append(vocab['<UNK>'])

    indexed_text = torch.tensor(indexed_text, dtype = torch.long).unsqueeze(0)
    indexed_text = indexed_text.to(device)

    # Evaluating Output and Converting Output to Probabilites
    output_logits = model(indexed_text)
    probs = nn.functional.softmax(output_logits, dim=1)

    # Showing Predicted Word:
    val, index = torch.max(probs, dim=1)
    predicted_word = list(vocab.keys())[index]
    test_text = test_text + " " + predicted_word

    return test_text

print("Input Text:",test_text)
for predictions in range(10):

    test_text = prediction(test_text)
    print(test_text)


Input Text: The College
the college started
the college started functioning
the college started functioning in
the college started functioning in 1960
the college started functioning in 1960 from
the college started functioning in 1960 from a
the college started functioning in 1960 from a camp
the college started functioning in 1960 from a camp office
the college started functioning in 1960 from a camp office in
the college started functioning in 1960 from a camp office in the
